# Chapter 2: working with text data

## 2.2 Tokenizing text


In [1]:
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt")
    filename = "the-verdict.txt"
    urllib.request.urlretrieve(url, filename)

In [2]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [3]:
raw_text

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [4]:
len(raw_text)

20479

In [5]:
import re

text = "Hello, worl. This, is a test"

result = re.split(r'(\s)', text)
result

['Hello,', ' ', 'worl.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test']

In [6]:
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'worl', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test']


In [7]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'worl', '.', 'This', ',', 'is', 'a', 'test']


In [8]:
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'worl', '.', 'This', ',', 'is', 'a', 'test']


In [9]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [10]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Converting tokens into token IDs

In [11]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [12]:
vocab = {token:integer for integer,token in enumerate(all_words)}

In [13]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


Modificación del original para crear el tokenizador directamente desde el texto, el preprocesamiento lo realiza internamente.

In [14]:
class SimpleTokenizerV1:
    """
    A simple tokenizer that builds its vocabulary from raw text.

    It preprocesses the input text, creates a vocabulary from the unique tokens,
    and provides methods to encode and decode text.
    """

    def __init__(self, raw_text: str) -> None:
        """
        Initialize the tokenizer from raw text.

        Args:
            raw_text: Text used to build the tokenizer vocabulary.
        """
        # Preprocess the raw text into tokens.
        preprocessed = self.preprocess(raw_text)

        # Get all unique tokens sorted alphabetically.
        self.all_words = sorted(set(preprocessed))

        # Store the vocabulary size.
        self.vocab_size = len(all_words)

        # Create the token-to-ID vocabulary.
        self.str_to_int = {
            token: integer
            for integer, token in enumerate(all_words)
        }

        # Create the ID-to-token vocabulary.
        self.int_to_str = {
            integer: token
            for token, integer in self.str_to_int.items()
        }

    def preprocess(self, text: str) -> list[str]:
        """
        Split text into clean tokens.

        Args:
            text: Input text.

        Returns:
            List of processed tokens.
        """
        # Split text using punctuation, double hyphens and whitespace.
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        # Remove empty tokens and surrounding whitespace.
        preprocessed = [
            item.strip()
            for item in preprocessed
            if item.strip()
        ]

        # Return the processed tokens.
        return preprocessed

    def encode(self, text: str) -> list[int]:
        """
        Convert text into a list of token IDs.

        Args:
            text: Input text.

        Returns:
            List of token IDs.
        """
        # Preprocess the input text into tokens.
        preprocessed = self.preprocess(text)

        # Convert each token into its vocabulary ID.
        ids = [self.str_to_int[token] for token in preprocessed]

        # Return the encoded sequence.
        return ids

    def decode(self, ids: list[int]) -> str:
        """
        Convert a list of token IDs back into text.

        Args:
            ids: List of token IDs.

        Returns:
            Decoded text.
        """
        # Convert each ID back to its token and join tokens with spaces.
        text = " ".join([self.int_to_str[index] for index in ids])

        # Remove spaces before punctuation marks.
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

        # Return the decoded text.
        return text

In [15]:
tokenizer = SimpleTokenizerV1(raw_text)

In [16]:
tokenizer.all_words[10:20], tokenizer.vocab_size

(['?', 'A', 'Ah', 'Among', 'And', 'Are', 'Arrt', 'As', 'At', 'Be'], 1130)

In [17]:
text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""

In [18]:
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [19]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [20]:
# esto deberia dar lo mismo que el texto original
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

## Adding special context tokens

Hello word is not in the orignal text data

In [22]:
text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

KeyError: 'Hello'

In [23]:
tokenizer.all_words.extend(["<|endoftext|>", "<|unk|>"])

In [24]:
len(tokenizer.all_words)

1132

Extendemos la clase para incorporar esos carácteres

In [25]:
class SimpleTokenizerV2:
    """
    A simple tokenizer that builds its vocabulary from raw text.

    It preprocesses the input text, creates a vocabulary from the unique tokens,
    and provides methods to encode and decode text.
    """

    def __init__(self, raw_text: str) -> None:
        """
        Initialize the tokenizer from raw text.

        Args:
            raw_text: Text used to build the tokenizer vocabulary.
        """
        # Preprocess the raw text into tokens.
        preprocessed = self.preprocess(raw_text)

        # Get all unique tokens sorted alphabetically.
        self.all_words = sorted(set(preprocessed))

        # Add special tokens to the vocabulary.
        self.all_words.extend(["<|endoftext|>", "<|unk|>"])

        # Store the vocabulary size.
        self.vocab_size = len(self.all_words)

        # Create the token-to-ID vocabulary.
        self.str_to_int = {
            token: integer
            for integer, token in enumerate(self.all_words)
        }

        # Create the ID-to-token vocabulary.
        self.int_to_str = {
            integer: token
            for token, integer in self.str_to_int.items()
        }

    def preprocess(self, text: str) -> list[str]:
        """
        Split text into clean tokens.

        Args:
            text: Input text.

        Returns:
            List of processed tokens.
        """
        # Split text using punctuation, double hyphens and whitespace.
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        # Remove empty tokens and surrounding whitespace.
        preprocessed = [
            item.strip()
            for item in preprocessed
            if item.strip()
        ]

        # Return the processed tokens.
        return preprocessed

    def encode(self, text: str) -> list[int]:
        """
        Convert text into a list of token IDs.

        Args:
            text: Input text.

        Returns:
            List of token IDs.
        """
        # Preprocess the input text into tokens.
        preprocessed = self.preprocess(text)

        # Replace unknown tokens with the unknown token.
        preprocessed = [
            token if token in self.str_to_int else "<|unk|>"
            for token in preprocessed
        ]

        # Convert each token into its vocabulary ID.
        ids = [self.str_to_int[token] for token in preprocessed]

        # Return the encoded sequence.
        return ids

    def decode(self, ids: list[int]) -> str:
        """
        Convert a list of token IDs back into text.

        Args:
            ids: List of token IDs.

        Returns:
            Decoded text.
        """
        # Convert each ID back to its token and join tokens with spaces.
        text = " ".join([self.int_to_str[index] for index in ids])

        # Remove spaces before punctuation marks.
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

        # Return the decoded text.
        return text

In [26]:
tokenizer = SimpleTokenizerV2(raw_text)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [27]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [28]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

## 2.5 Byte pair encoding

In [29]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.13.0


In [30]:
tokenizer = tiktoken.get_encoding("gpt2")

In [31]:
tokenizer.encode("Hello world")

[15496, 995]

In [32]:
tokenizer.decode(tokenizer.encode("Hello world"))

'Hello world'

In [33]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [34]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


## 2.6 Data sampling with with sliding window

In [35]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [36]:
enc_sample = enc_text[50:]

In [37]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [38]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [39]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [40]:
import torch
print("PyTorch version:", torch.__version__)

import torch
from torch.utils.data import Dataset, DataLoader

PyTorch version: 2.12.1


In [41]:
class GPTDatasetV1(Dataset):
    """
    Dataset for preparing tokenized text sequences for GPT-style training.

    It creates input-target pairs using a sliding window over the full text.
    The target sequence is the input sequence shifted one token to the right.
    """

    def __init__(self, text, tokenizer, max_length, stride):
        """
        Initialize the dataset.

        Args:
            txt: Raw input text.
            tokenizer: Tokenizer used to convert text into token IDs.
            max_length: Number of tokens in each input sequence.
            stride: Step size used to move the sliding window.
        """
        # Store the input sequences.
        self.input_ids = []

        # Store the target sequences.
        self.target_ids = []

        # Tokenize the entire text.
        token_ids = tokenizer.encode(
            text,
            allowed_special={"<|endoftext|>"}
        )

        # Ensure the text is long enough to create at least one training sample.
        assert len(token_ids) > max_length, (
            "Number of tokenized inputs must at least be equal to "
            "max_length + 1"
        )

        # Create overlapping input-target chunks using a sliding window.
        for i in range(0, len(token_ids) - max_length, stride):

            # Select max_length tokens as the model input.
            input_chunk = token_ids[i:i + max_length]

            # Select the same sequence shifted one token to the right.
            target_chunk = token_ids[i + 1:i + max_length + 1]

            # Convert the input chunk to a PyTorch tensor and store it.
            self.input_ids.append(torch.tensor(input_chunk))

            # Convert the target chunk to a PyTorch tensor and store it.
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        """
        Return the number of training samples in the dataset.

        Returns:
            Number of input-target pairs.
        """
        # Return the number of stored input sequences.
        return len(self.input_ids)

    def __getitem__(self, idx):
        """
        Return one training sample.

        Args:
            idx: Index of the sample.

        Returns:
            Tuple with input IDs and target IDs.
        """
        # Return the input sequence and its corresponding target sequence.
        return self.input_ids[idx], self.target_ids[idx]

In [42]:
# Initialize the tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# Create dataset
dataset = GPTDatasetV1(text = raw_text, 
                       tokenizer = tokenizer, 
                       max_length=4, 
                       stride = 1)

# Create dataloader
dataloader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=False,
    drop_last=True,
    num_workers=0
)


In [43]:
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257],
        [10899,  2138,   257,  7026]])


## 2.7 Creating token embeddings

## 2.8 Encoding word positions

In [44]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [45]:
# Initialize the tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# Create dataset
dataset = GPTDatasetV1(text = raw_text, 
                       tokenizer = tokenizer, 
                       max_length=4, 
                       stride = 4)

# Create dataloader
dataloader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=False,
    drop_last=True,
    num_workers=0
)

In [46]:
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [47]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


embeddings de los tokens, pasan el vector de tokens al espacio

In [48]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [49]:
context_length = 4 # max lenght
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [50]:
pos_embeddings = pos_embedding_layer(torch.arange(4))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [51]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
